In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('data.csv')

cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)

num_cols = df.select_dtypes(include=['number']).columns.drop('ID')

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(df_train[num_cols])
X_val_scaled = scaler.transform(df_val[num_cols])

imputer = KNNImputer(n_neighbors=5)
X_train_imputed_scaled = imputer.fit_transform(X_train_scaled)
X_val_imputed_scaled = imputer.transform(X_val_scaled)

X_train_imputed = scaler.inverse_transform(X_train_imputed_scaled)
X_val_imputed = scaler.inverse_transform(X_val_imputed_scaled)

df_train_imputed = pd.DataFrame(X_train_imputed, columns=num_cols, index=df_train.index)
df_val_imputed = pd.DataFrame(X_val_imputed, columns=num_cols, index=df_val.index)

if 'Stage' in num_cols:
    df_train_imputed['Stage'] = df_train_imputed['Stage'].round()
    df_val_imputed['Stage'] = df_val_imputed['Stage'].round()

df_train_final = df_train.copy()
df_val_final = df_val.copy()

for col in num_cols:
    df_train_final[col] = df_train_final[col].fillna(df_train_imputed[col])
    df_val_final[col] = df_val_final[col].fillna(df_val_imputed[col])

df_train_final.to_csv('pbc_train_imputed.csv', index=False)
df_val_final.to_csv('pbc_val_imputed.csv', index=False)

W kolumnach kategorycznych dodaje wartość "unknown" jako osobną klasę wartości. Natomiast dla cech numerycznych używam knn

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train_raw_num = df_train_final[num_cols].values
X_val_raw_num = df_val_final[num_cols].values

cat_features = [c for c in cat_cols if c != 'Status']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(df_train_final[cat_features])
X_val_cat = encoder.transform(df_val_final[cat_features])

y_train = df_train_final['Status'].values
y_val = df_val_final['Status'].values

X_train_baseline = np.hstack([X_train_raw_num, X_train_cat])
X_val_baseline = np.hstack([X_val_raw_num, X_val_cat])

results = {}

def evaluate(X_tr, X_va, name):
    lr = LogisticRegression(solver='newton-cg', random_state=42, max_iter=1000)
    lr.fit(X_tr, y_train)
    acc_lr = accuracy_score(y_val, lr.predict(X_va))
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_tr, y_train)
    acc_dt = accuracy_score(y_val, dt.predict(X_va))
    results[name] = {'Regresja Logistyczna': acc_lr, 'Drzewo Decyzyjne': acc_dt}

evaluate(X_train_baseline, X_val_baseline, 'Bez przetwarzania (Baseline)')

norm = MinMaxScaler()
X_tr_norm = np.hstack([norm.fit_transform(X_train_raw_num), X_train_cat])
X_va_norm = np.hstack([norm.transform(X_val_raw_num), X_val_cat])
evaluate(X_tr_norm, X_va_norm, 'Normalizacja')

std = StandardScaler()
X_tr_std_num = std.fit_transform(X_train_raw_num)
X_va_std_num = std.transform(X_val_raw_num)
X_tr_std = np.hstack([X_tr_std_num, X_train_cat])
X_va_std = np.hstack([X_va_std_num, X_val_cat])
evaluate(X_tr_std, X_va_std, 'Standaryzacja')

disc = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform', subsample=None)
X_tr_disc = np.hstack([disc.fit_transform(X_train_raw_num), X_train_cat])
X_va_disc = np.hstack([disc.transform(X_val_raw_num), X_val_cat])
evaluate(X_tr_disc, X_va_disc, 'Dyskretyzacja')

sel = SelectKBest(score_func=f_classif, k=10)
X_tr_sel = sel.fit_transform(X_train_baseline, y_train)
X_va_sel = sel.transform(X_val_baseline)
evaluate(X_tr_sel, X_va_sel, 'Selekcja cech')

pca = PCA(n_components=5, random_state=42)
X_tr_pca = np.hstack([pca.fit_transform(X_tr_std_num), X_train_cat])
X_va_pca = np.hstack([pca.transform(X_va_std_num), X_val_cat])
evaluate(X_tr_pca, X_va_pca, 'PCA')

res_df = pd.DataFrame(results).T
res_df

,Regresja Logistyczna,Drzewo Decyzyjne
Bez przetwarzania (Baseline),0.785714,0.607143
Normalizacja,0.833333,0.607143
Standaryzacja,0.785714,0.607143
Dyskretyzacja,0.797619,0.750000
Selekcja cech,0.797619,0.738095
PCA,0.785714,0.654762


TODO WNIOSKI


In [4]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import pandas as pd

nb_classifier = GaussianNB()
nb_param_grid = {
    'var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]
}

nb_grid_search = GridSearchCV(estimator=nb_classifier, param_grid=nb_param_grid, cv=5, scoring='accuracy')
nb_grid_search.fit(X_tr_std, y_train)

dt_classifier = DecisionTreeClassifier(random_state=42)
dt_param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': [2, 5, 10, 20]
}

dt_grid_search = GridSearchCV(estimator=dt_classifier, param_grid=dt_param_grid, cv=5, scoring='accuracy')
dt_grid_search.fit(X_tr_std, y_train)

nb_best_model = nb_grid_search.best_estimator_
dt_best_model = dt_grid_search.best_estimator_

nb_val_predictions = nb_best_model.predict(X_va_std)
dt_val_predictions = dt_best_model.predict(X_va_std)

nb_val_accuracy = accuracy_score(y_val, nb_val_predictions)
dt_val_accuracy = accuracy_score(y_val, dt_val_predictions)

tuning_results = pd.DataFrame({
    'Algorytm': ['Naiwny klasyfikator Bayesa', 'Drzewo decyzyjne'],
    'Najlepsze hiperparametry': [str(nb_grid_search.best_params_), str(dt_grid_search.best_params_)],
    'Dokładność (Zbiór walidacyjny)': [nb_val_accuracy, dt_val_accuracy]
})

tuning_results

,Algorytm,Najlepsze hiperparametry,Dokładność (Zbiór walidacyjny)
0,Naiwny klasyfikator Bayesa,{'var_smoothing': 0.1},0.726190
1,Drzewo decyzyjne,"{'criterion': 'entropy', 'max_depth': 5, 'min_...",0.714286
